# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY")
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

@tool
def retrieve_game(query: str) -> str:
    """Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=5)
    docs = []
    for metadata in results["metadatas"][0]:
        docs.append({
            "Name": metadata.get("Name"),
            "Platform": metadata.get("Platform"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
        })
    return json.dumps(docs)

#### Evaluate Retrieval Tool

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini")
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"User question: {question}\n\n"
        f"Retrieved documents:\n{retrieved_docs}\n\n"
        "Respond with a JSON object with two fields:\n"
        '- "useful": true if the documents contain enough information to answer the question, false otherwise\n'
        '- "description": a detailed explanation of your evaluation'
    )
    response = llm.invoke(prompt)
    try:
        report = EvaluationReport.model_validate_json(response.content)
    except Exception:
        # Fallback: parse manually if JSON extraction needed
        content = response.content or ""
        useful = "true" in content.lower() and "false" not in content.lower()
        report = EvaluationReport(useful=useful, description=content)
    return report.model_dump_json()

#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str) -> str:
    """Search the web for video game industry information.
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query=question, max_results=5)
    results = [
        {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
        for r in response.get("results", [])
    ]
    return json.dumps(results)

### Agent

In [ ]:
instructions = """You are UdaPlay, an expert AI research agent for the video game industry.

Your goal is to answer questions about video games accurately and helpfully.

Follow this workflow for every question:
1. Use `retrieve_game` to search the local vector database for relevant game information.
2. Use `evaluate_retrieval` to assess whether the retrieved results are sufficient to answer the question.
3. If the evaluation says the documents are NOT useful, use `game_web_search` to find the answer online.
4. Provide a clear, concise, and accurate final answer to the user.

Always base your answer on the best available information from the tools.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

In [ ]:
queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for query in queries:
    print(f"\nQ: {query}")
    run = agent.invoke(query)
    final_state = run.get_final_state()
    messages = final_state["messages"]
    # Get the last AIMessage with content as the final answer
    answer = next(
        (m.content for m in reversed(messages) if isinstance(m, AIMessage) and m.content),
        "No answer found."
    )
    print(f"A: {answer}")

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes